In [ ]:
%%capture
!pip install --upgrade scikit-learn
!pip install --upgrade gensim
!apt-get install git

In [ ]:
%%capture
!git clone https://github.com/YJiangcm/SST-2-sentiment-analysis.git

In [ ]:
import re
import torch
import torch.nn as nn
import torch.nn.functional as F

def Preprocess(Text):
    msg = re.sub('[^a-zA-Z]', ' ', Text)
    msg = msg.lower()
    msg = msg.split()
    msg = ' '.join(msg)
    return msg

In [ ]:
#FTX = gensim.downloader.load('fasttext-wiki-news-subwords-300')

In [ ]:
trainset = pd.read_csv("SST-2-sentiment-analysis/data/train.tsv", sep='\t', header=None, names=["class", "review"])
testset = pd.read_csv("SST-2-sentiment-analysis/data/test.tsv", sep='\t', header=None, names=["class", "review"])
trainset["p-review"] = trainset["review"].apply(Preprocess)
testset["p-review"] = testset["review"].apply(Preprocess)

In [ ]:
def splitClsData(dset):
  POS=[]
  NEG=[]
  VOCAB = {"<pad>": 0, "<s>": 1, "</s>": 2, "<unk>": 3}
  nVOCAB = 4
  for idx in range(len(dset)):
    words = dset["p-review"][idx].split()
    for word in words:
      if VOCAB.get(word) == None:
        VOCAB[word] = nVOCAB
        nVOCAB += 1
    words = ['<s>'] + words + ['</s>']
    if trainset["class"][idx] == 0:
      POS.append(words)
    else:
      NEG.append(words)
  return POS, NEG, VOCAB

def tokenize(words, VOCAB):
  ids = []
  for word in words:
    if VOCAB.get(word) == None:
      ids.append(VOCAB["<unk>"])
    else:
      ids.append(VOCAB[word])
  return ids

def batchSamples(samples, VOCAB, bsize=16):
  batches = []
  for i in range(len(samples) // bsize):
    maxlen = 0
    for j in range(bsize):
      l = len(samples[i*bsize + j])
      if maxlen < l:
        maxlen = l
    batch = np.zeros((bsize, maxlen))
    for j in range(bsize):
      l = len(samples[i*bsize + j])
      ids = samples[i*bsize + j]
      if l < maxlen:
         ids += [VOCAB["<pad>"]] * (maxlen - l)
      batch[j] = ids
    batches.append(batch)
  return batches

In [ ]:
class OneHotRNN(nn.Module):

    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size

        self.Wx = nn.Linear(vocab_size, hidden_size)
        self.Wh = nn.Linear(hidden_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.vsize = vocab_size

    def forward(self, x, h=None):

        batch_size, seq_len = x.shape

        if h is None:
            h = torch.zeros(batch_size, self.hidden_size, device=x.device)

        outputs = []

        for t in range(seq_len):
            xt = F.one_hot(x[:, t], self.vsize).to(torch.float)
            h = torch.tanh(
                self.Wx(xt) + self.Wh(h)
            )

            out = self.fc(h)

            outputs.append(out)

        outputs = torch.stack(outputs, dim=1)

        return outputs, h

class EmbRNN(nn.Module):

    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embed_size)

        self.Wx = nn.Linear(embed_size, hidden_size)
        self.Wh = nn.Linear(hidden_size, hidden_size)

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):

        batch_size, seq_len = x.shape

        if h is None:
            h = torch.zeros(batch_size, self.hidden_size, device=x.device)

        x = self.embedding(x)

        outputs = []

        for t in range(seq_len):

            xt = x[:, t, :]

            h = torch.tanh(
                self.Wx(xt) + self.Wh(h)
            )

            out = self.fc(h)

            outputs.append(out)

        outputs = torch.stack(outputs, dim=1)

        return outputs, h

In [ ]:
#dim = FTX.vector_size

POSSents, NEGSents, VOCAB = splitClsData(trainset)

LOOKUP = {}
for word in VOCAB:
  LOOKUP[VOCAB[word]] = word



POS = []
NEG = []

for sent in POSSents:
  POS.append(tokenize(sent, VOCAB))

for sent in NEGSents:
  NEG.append(tokenize(sent, VOCAB))

POSBatches = batchSamples(POS, VOCAB)
NEGBatches = batchSamples(NEG, VOCAB)

In [ ]:
device = "cuda"

vocab_size = len(VOCAB)
model = OneHotRNN(vocab_size, 256).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

for epoch in range(20):

    total_loss = 0

    for batch in POSBatches:

        x = torch.LongTensor(batch[:,:-1]).to(device)
        y = torch.LongTensor(batch[:,1:]).to(device)

        logits, _ = model(x)

        loss = criterion(
            logits.view(-1, vocab_size),
            y.view(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("epoch", epoch, total_loss)

epoch 0 1552.7783489227295
epoch 1 1403.3539028167725
epoch 2 1351.7758932113647
epoch 3 1315.7046575546265
epoch 4 1314.7514429092407
epoch 5 1294.3623752593994
epoch 6 1263.717758178711
epoch 7 1235.6729459762573
epoch 8 1232.0431971549988
epoch 9 1204.5652952194214
epoch 10 1165.4662590026855
epoch 11 1182.7104649543762
epoch 12 1202.1149673461914
epoch 13 1180.7542152404785
epoch 14 1139.6880712509155
epoch 15 1188.7477536201477
epoch 16 1154.594539642334
epoch 17 1178.4361424446106
epoch 18 1163.3221321105957
epoch 19 1130.5549240112305


In [ ]:
import random

def selectID(arr, topk=50):
  sel = random.random()
  toparr = sorted(((val, idx) for idx, val in enumerate(arr)),reverse=True)[:topk]
  ex = [p for p, _ in toparr]
  ex = np.exp(ex)
  prob = ex/np.sum(ex)
  b = 0
  nwords = len(toparr)
  for i in range(nwords):
    b += prob[i]
    prob[i] = b
  b = 0
  id = 0
  for i in range(nwords):
    if sel > b and sel <= prob[i]:
      id = i
      break;
    b = prob[i]
  return toparr[id][1]

def generate(model, VOCAB, LOOKUP, topk=50):
  genText = []
  sents = []

  vocabSize = len(VOCAB)
  id = VOCAB["<s>"]

  x = torch.LongTensor([[id]]).to(device)

  out, h = model(x)

  id = selectID(out[0].detach().cpu().numpy()[0], topk=topk)
  predWord = LOOKUP[id]
  genText.append(predWord)
  if predWord == "</s>":
    return genText
  predX = torch.LongTensor([[id]]).to(device)
  for i in range(80):
    out, h = model(predX, h)
    id = selectID(out[0].detach().cpu().numpy()[0], topk=topk)
    predWord = LOOKUP[id]
    predX = torch.LongTensor([[id]]).to(device)
    if predWord == "</s>":
      return genText
    genText.append(predWord)

  return genText

In [ ]:
for i in range(20):
  words = generate(model, VOCAB, LOOKUP)
  print(" ".join(words))

the film portrays lacking is so large talky nor all that s film for competent every version of marivaux s instance one of whimsy
the dramatic thing you ll endured the latter raunchy stinker
i admire you have you very a awkward of gripping and his homage
it does not so long and the film and the other mell cartoons things that he and much of the level of counterculture in a sense of this amp and ben with the sequel s trademark as the director that depicts the guitar of their assigned year
this is rote is that celebi in an hour
a movie
plays in trying to imagine the insipid director and an animatronic nor a disastrous car and a hubert warmed of intelligibility
even the story prowess and spits on the director and pub and fiendish acts
but the striking version with a toddler
this is an film that s too good from this distinguished the mill ocean and a year generation and special that meanderings than not another a movie that liman versus personal attitude in jennifer actress it s just two mo

In [ ]:
emodel = EmbRNN(vocab_size, 256, 256).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(emodel.parameters(), lr=0.003)

for epoch in range(20):

    total_loss = 0

    for batch in POSBatches:

        x = torch.LongTensor(batch[:,:-1]).to(device)
        y = torch.LongTensor(batch[:,1:]).to(device)

        logits, _ = emodel(x)

        loss = criterion(
            logits.view(-1, vocab_size),
            y.view(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("epoch", epoch, total_loss)

epoch 0 1448.3379483222961
epoch 1 1210.0884652137756
epoch 2 1052.2832884788513
epoch 3 915.0457410812378
epoch 4 807.8469746112823
epoch 5 729.2327725887299
epoch 6 664.8492732048035
epoch 7 609.6696181297302
epoch 8 557.1680474281311
epoch 9 512.8773180246353
epoch 10 477.14408552646637
epoch 11 448.08746790885925
epoch 12 423.2379254102707
epoch 13 412.68154537677765
epoch 14 397.3337961435318
epoch 15 385.6379861831665
epoch 16 381.8469465970993
epoch 17 369.76753520965576
epoch 18 355.35099160671234
epoch 19 342.3559386730194


In [ ]:
for i in range(20):
  words = generate(emodel, VOCAB, LOOKUP)
  print(" ".join(words))

you might be thinking about or not feel after a half hour
this new zealand dialogue the sober minded and the high original nor terribly much more about this film it s left his sense of suspense
it s too close to no fewer story is about cinema that seem the action movie esque affected child acting like the wanderers and most
this is rote drivel aimed at mom and dad the dreams of youth should remain all that funny
a depraved incoherent instantly disposable piece day
more busy than exciting as thinking man cia agent jack ryan in this wildly uneven and inconsistent margarita happy the thoroughly formulaic and makes sorry use of aaliyah in her one
the results are far too much of the movie is ugly pointless
one sided documentary offers simplistic explanations to a bad improvisation movie would ever knew what i can analyze this movie
the movie is just a film which presses familiar herzog tropes into his future
the story is n t nearly as captivating as telling as the war scenes true to the law